In [ ]:
print ("بسم الله والحمدلله والصلاة والسلام على رسول الله صلى الله عليه وسلم")

بسم الله والحمدلله والصلاة والسلام على رسول الله صلى الله عليه وسلم


In [ ]:
# ============================================================
# 1. INSTALL / IMPORT LIBRARIES
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from collections import Counter

In [ ]:

from statsmodels.stats.contingency_tables import mcnemar
from tabulate import tabulate

In [ ]:
#connect to the drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
ANER_FILES={
"Camel-MSA": "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_msa_predictions_CANER_Corpus.txt",
"Camel-CA":     "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_ca_predictions_CANER_Corpus.txt",
"Camel-Mix": "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_mix_predictions_CANER_Corpus.txt",
"Arbert": "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Arbert_predictions_CANER_Corpus.txt",
 "Ensemble-Majority" : "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Compined_Predictions_Majority_CANER.txt",
# "Ensemble-Weighted" : "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Compined_Predictions_Weighted_CANER.txt"
 "Ensemble-Weighted" : "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/New_CWeighted_Caner.txt"

             }

#New McNemar Test For Majority Voting

In [ ]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from tabulate import tabulate


# ============================================================
# 1. Load Prediction Files Safely
# ============================================================

def load_predictions(file_path):
    """
    Safely reads NER prediction files containing:

    Token    Gold Tag    Predicted Tag

    The file may contain:
    - quotation marks (")
    - blank lines
    - extra trailing tabs

    Only the first three tab-separated fields are used.
    """

    rows = []
    extra_field_lines = []

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:

        for line_number, line in enumerate(f, start=1):

            # Skip header
            if line_number == 1:
                continue

            # Remove line ending
            line = line.rstrip("\r\n")

            # Skip completely empty lines
            if not line.strip():
                continue

            # Split using TAB
            parts = line.split("\t")

            # Need at least 3 fields
            if len(parts) < 3:
                print(
                    f"⚠️ Skipping malformed line {line_number} "
                    f"in {file_path}: {parts}"
                )
                continue

            # Record lines having extra fields
            if len(parts) > 3:
                extra_field_lines.append(line_number)

            # Take only the first three fields
            token = parts[0]
            gold = parts[1]
            predicted = parts[2]

            rows.append([token, gold, predicted])

    df = pd.DataFrame(
        rows,
        columns=["Token", "Gold Tag", "Predicted Tag"]
    )

    print(
        f"✅ Loaded {len(df)} predictions from "
        f"{file_path.split('/')[-1]}"
    )

    if extra_field_lines:
        print(
            f"   ⚠️ {len(extra_field_lines)} lines had extra "
            f"trailing fields; first 3 fields were used."
        )

    return df.reset_index(drop=True)


# ============================================================
# 2. Files
# ============================================================

baseline_files = [

  "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_msa_predictions_CANER_Corpus.txt",
    "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_ca_predictions_CANER_Corpus.txt",
 "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_mix_predictions_CANER_Corpus.txt",
 "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Arbert_predictions_CANER_Corpus.txt"
 ]


# ============================================================
# Proposed Ensemble
# ============================================================

ensemble_file = (
  "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Compined_Predictions_Majority_CANER.txt"

)


# ============================================================
# 3. Load Ensemble
# ============================================================

ensemble = load_predictions(ensemble_file)


print("\n")
print("=" * 90)
print("McNemar Test: Baseline Models vs. Proposed Ensemble")
print("=" * 90)

print(
    f"\nEnsemble: {ensemble_file.split('/')[-1]}"
)

print(
    f"Number of ensemble predictions: {len(ensemble)}"
)


# ============================================================
# 4. Results Containers
# ============================================================

results_summary = []
tables = {}


# ============================================================
# 5. Compare Each Baseline with Ensemble
# ============================================================

for file in baseline_files:

    model_name = file.split("/")[-1]

    print("\n" + "-" * 90)
    print(f"Processing: {model_name}")
    print("-" * 90)

    # --------------------------------------------------------
    # Load baseline
    # --------------------------------------------------------

    baseline = load_predictions(file)

    print(
        f"Baseline predictions : {len(baseline)}"
    )

    print(
        f"Ensemble predictions : {len(ensemble)}"
    )


    # ========================================================
    # 6. Verify Number of Predictions
    # ========================================================

    if len(baseline) != len(ensemble):

        raise ValueError(
            f"\n❌ Number of predictions does not match!\n"
            f"Baseline = {len(baseline)}\n"
            f"Ensemble = {len(ensemble)}\n"
            f"File = {model_name}"
        )


    # ========================================================
    # 7. Verify Token Alignment
    # ========================================================

    token_match = baseline["Token"].equals(
        ensemble["Token"]
    )

    if not token_match:

        mismatch = baseline["Token"] != ensemble["Token"]

        indices = mismatch[mismatch].index[:10]

        print("\n❌ Token alignment error!")

        for idx in indices:

            print(
                f"Row {idx}: "
                f"Baseline = {repr(baseline.loc[idx, 'Token'])} | "
                f"Ensemble = {repr(ensemble.loc[idx, 'Token'])}"
            )

        raise ValueError(
            f"\n❌ Token order/content does not match "
            f"between baseline and ensemble:\n{model_name}"
        )

    print("✅ Token alignment verified")


    # ========================================================
    # 8. Verify Gold Tag Alignment
    # ========================================================

    gold_match = baseline["Gold Tag"].equals(
        ensemble["Gold Tag"]
    )

    if not gold_match:

        mismatch = baseline["Gold Tag"] != ensemble["Gold Tag"]

        indices = mismatch[mismatch].index[:10]

        print("\n❌ Gold Tag alignment error!")

        for idx in indices:

            print(
                f"Row {idx}: "
                f"Baseline Gold = "
                f"{repr(baseline.loc[idx, 'Gold Tag'])} | "
                f"Ensemble Gold = "
                f"{repr(ensemble.loc[idx, 'Gold Tag'])}"
            )

        raise ValueError(
            f"\n❌ Gold Tags do not match "
            f"between baseline and ensemble:\n{model_name}"
        )

    print("✅ Gold Tag alignment verified")


    # ========================================================
    # 9. Determine Correct / Incorrect Predictions
    # ========================================================

    baseline_correct = (
        baseline["Gold Tag"] ==
        baseline["Predicted Tag"]
    )

    ensemble_correct = (
        ensemble["Gold Tag"] ==
        ensemble["Predicted Tag"]
    )


    # ========================================================
    # 10. McNemar 2×2 Table
    # ========================================================

    # Both models are correct
    a = int(
        (baseline_correct & ensemble_correct).sum()
    )

    # Baseline correct, Ensemble wrong
    b = int(
        (baseline_correct & ~ensemble_correct).sum()
    )

    # Baseline wrong, Ensemble correct
    c = int(
        (~baseline_correct & ensemble_correct).sum()
    )

    # Both models are wrong
    d = int(
        (~baseline_correct & ~ensemble_correct).sum()
    )


    table = [
        [a, b],
        [c, d]
    ]

    tables[model_name] = table


    # ========================================================
    # 11. McNemar Test
    # ========================================================

    result = mcnemar(
        table,
        exact=False,
        correction=True
    )


    # ========================================================
    # 12. Significance
    # ========================================================

    if result.pvalue < 0.05:

        significance = "Significant (p < 0.05)"

    else:

        significance = "Not significant"


    # ========================================================
    # 13. Store Results
    # ========================================================

    results_summary.append([
        model_name,
        a,
        b,
        c,
        d,
        result.statistic,
        result.pvalue,
        significance
    ])


# ============================================================
# 14. Final Results
# ============================================================

print("\n\n")
print("=" * 110)
print("McNemar Test Results")
print("=" * 110)

headers = [
    "Baseline Model",
    "a",
    "b",
    "c",
    "d",
    "Chi² Statistic",
    "p-value",
    "Significance"
]

print(
    tabulate(
        results_summary,
        headers=headers,
        tablefmt="grid",
        floatfmt=".5f"
    )
)


# ============================================================
# 15. Detailed McNemar Tables
# ============================================================

print("\n\n")
print("=" * 90)
print("Detailed 2×2 McNemar Comparison Tables")
print("=" * 90)

for model_name, table in tables.items():

    print(f"\n{model_name}:")

    print(
        tabulate(
            table,
            headers=[
                "Ensemble Correct",
                "Ensemble Wrong"
            ],
            showindex=[
                "Baseline Correct",
                "Baseline Wrong"
            ],
            tablefmt="fancy_grid"
        )
    )


# ============================================================
# 16. Discordant Cases
# ============================================================

print("\n\n")
print("=" * 90)
print("Discordant Cases")
print("=" * 90)

for row in results_summary:

    model_name = row[0]

    b = row[2]
    c = row[3]

    print(f"\n{model_name}")

    print(
        f"Baseline correct / Ensemble wrong = {b}"
    )

    print(
        f"Baseline wrong / Ensemble correct = {c}"
    )

    if c > b:

        print(
            "→ Ensemble corrects more errors than it introduces."
        )

    elif b > c:

        print(
            "→ Ensemble introduces more errors than it corrects."
        )

    else:

        print(
            "→ Both have the same number of discordant cases."
        )

✅ Loaded 24966 predictions from Compined_Predictions_Majority_CANER.txt


McNemar Test: Baseline Models vs. Proposed Ensemble

Ensemble: Compined_Predictions_Majority_CANER.txt
Number of ensemble predictions: 24966

------------------------------------------------------------------------------------------
Processing: Camel_msa_predictions_CANER_Corpus.txt
------------------------------------------------------------------------------------------
✅ Loaded 24966 predictions from Camel_msa_predictions_CANER_Corpus.txt
Baseline predictions : 24966
Ensemble predictions : 24966
✅ Token alignment verified
✅ Gold Tag alignment verified

------------------------------------------------------------------------------------------
Processing: Camel_ca_predictions_CANER_Corpus.txt
------------------------------------------------------------------------------------------
✅ Loaded 24966 predictions from Camel_ca_predictions_CANER_Corpus.txt
Baseline predictions : 24966
Ensemble predictions : 24966
✅ To

#New McNemar Test For Weighted Voting

In [ ]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from tabulate import tabulate


# ============================================================
# 1. Load Prediction Files Safely
# ============================================================

def load_predictions(file_path):
    """
    Safely reads NER prediction files containing:

    Token    Gold Tag    Predicted Tag

    The file may contain:
    - quotation marks (")
    - blank lines
    - extra trailing tabs

    Only the first three tab-separated fields are used.
    """

    rows = []
    extra_field_lines = []

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:

        for line_number, line in enumerate(f, start=1):

            # Skip header
            if line_number == 1:
                continue

            # Remove line ending
            line = line.rstrip("\r\n")

            # Skip completely empty lines
            if not line.strip():
                continue

            # Split using TAB
            parts = line.split("\t")

            # Need at least 3 fields
            if len(parts) < 3:
                print(
                    f"⚠️ Skipping malformed line {line_number} "
                    f"in {file_path}: {parts}"
                )
                continue

            # Record lines having extra fields
            if len(parts) > 3:
                extra_field_lines.append(line_number)

            # Take only the first three fields
            token = parts[0]
            gold = parts[1]
            predicted = parts[2]

            rows.append([token, gold, predicted])

    df = pd.DataFrame(
        rows,
        columns=["Token", "Gold Tag", "Predicted Tag"]
    )

    print(
        f"✅ Loaded {len(df)} predictions from "
        f"{file_path.split('/')[-1]}"
    )

    if extra_field_lines:
        print(
            f"   ⚠️ {len(extra_field_lines)} lines had extra "
            f"trailing fields; first 3 fields were used."
        )

    return df.reset_index(drop=True)


# ============================================================
# 2. Files
# ============================================================
baseline_files = [

  "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_msa_predictions_CANER_Corpus.txt",
    "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_ca_predictions_CANER_Corpus.txt",
 "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Camel_mix_predictions_CANER_Corpus.txt",
 "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Arbert_predictions_CANER_Corpus.txt"
 ]


# ============================================================
# Proposed Ensemble
# ============================================================

ensemble_file = (
# "Ensemble-Majority" : "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/Compined_Predictions_Majority_CANER.txt",
 "/content/drive/MyDrive/Ontology Learning 2023/paper1 (pre trained models for ANER)/Code/test on CANERCorpus/Predictions_CANER Corpus/New_CWeighted_Caner.txt"

)


# ============================================================
# 3. Load Ensemble
# ============================================================

ensemble = load_predictions(ensemble_file)


print("\n")
print("=" * 90)
print("McNemar Test: Baseline Models vs. Proposed Ensemble")
print("=" * 90)

print(
    f"\nEnsemble: {ensemble_file.split('/')[-1]}"
)

print(
    f"Number of ensemble predictions: {len(ensemble)}"
)


# ============================================================
# 4. Results Containers
# ============================================================

results_summary = []
tables = {}


# ============================================================
# 5. Compare Each Baseline with Ensemble
# ============================================================

for file in baseline_files:

    model_name = file.split("/")[-1]

    print("\n" + "-" * 90)
    print(f"Processing: {model_name}")
    print("-" * 90)

    # --------------------------------------------------------
    # Load baseline
    # --------------------------------------------------------

    baseline = load_predictions(file)

    print(
        f"Baseline predictions : {len(baseline)}"
    )

    print(
        f"Ensemble predictions : {len(ensemble)}"
    )


    # ========================================================
    # 6. Verify Number of Predictions
    # ========================================================

    if len(baseline) != len(ensemble):

        raise ValueError(
            f"\n❌ Number of predictions does not match!\n"
            f"Baseline = {len(baseline)}\n"
            f"Ensemble = {len(ensemble)}\n"
            f"File = {model_name}"
        )


    # ========================================================
    # 7. Verify Token Alignment
    # ========================================================

    token_match = baseline["Token"].equals(
        ensemble["Token"]
    )

    if not token_match:

        mismatch = baseline["Token"] != ensemble["Token"]

        indices = mismatch[mismatch].index[:10]

        print("\n❌ Token alignment error!")

        for idx in indices:

            print(
                f"Row {idx}: "
                f"Baseline = {repr(baseline.loc[idx, 'Token'])} | "
                f"Ensemble = {repr(ensemble.loc[idx, 'Token'])}"
            )

        raise ValueError(
            f"\n❌ Token order/content does not match "
            f"between baseline and ensemble:\n{model_name}"
        )

    print("✅ Token alignment verified")


    # ========================================================
    # 8. Verify Gold Tag Alignment
    # ========================================================

    gold_match = baseline["Gold Tag"].equals(
        ensemble["Gold Tag"]
    )

    if not gold_match:

        mismatch = baseline["Gold Tag"] != ensemble["Gold Tag"]

        indices = mismatch[mismatch].index[:10]

        print("\n❌ Gold Tag alignment error!")

        for idx in indices:

            print(
                f"Row {idx}: "
                f"Baseline Gold = "
                f"{repr(baseline.loc[idx, 'Gold Tag'])} | "
                f"Ensemble Gold = "
                f"{repr(ensemble.loc[idx, 'Gold Tag'])}"
            )

        raise ValueError(
            f"\n❌ Gold Tags do not match "
            f"between baseline and ensemble:\n{model_name}"
        )

    print("✅ Gold Tag alignment verified")


    # ========================================================
    # 9. Determine Correct / Incorrect Predictions
    # ========================================================

    baseline_correct = (
        baseline["Gold Tag"] ==
        baseline["Predicted Tag"]
    )

    ensemble_correct = (
        ensemble["Gold Tag"] ==
        ensemble["Predicted Tag"]
    )


    # ========================================================
    # 10. McNemar 2×2 Table
    # ========================================================

    # Both models are correct
    a = int(
        (baseline_correct & ensemble_correct).sum()
    )

    # Baseline correct, Ensemble wrong
    b = int(
        (baseline_correct & ~ensemble_correct).sum()
    )

    # Baseline wrong, Ensemble correct
    c = int(
        (~baseline_correct & ensemble_correct).sum()
    )

    # Both models are wrong
    d = int(
        (~baseline_correct & ~ensemble_correct).sum()
    )


    table = [
        [a, b],
        [c, d]
    ]

    tables[model_name] = table


    # ========================================================
    # 11. McNemar Test
    # ========================================================

    result = mcnemar(
        table,
        exact=False,
        correction=True
    )


    # ========================================================
    # 12. Significance
    # ========================================================

    if result.pvalue < 0.05:

        significance = "Significant (p < 0.05)"

    else:

        significance = "Not significant"


    # ========================================================
    # 13. Store Results
    # ========================================================

    results_summary.append([
        model_name,
        a,
        b,
        c,
        d,
        result.statistic,
        result.pvalue,
        significance
    ])


# ============================================================
# 14. Final Results
# ============================================================

print("\n\n")
print("=" * 110)
print("McNemar Test Results")
print("=" * 110)

headers = [
    "Baseline Model",
    "a",
    "b",
    "c",
    "d",
    "Chi² Statistic",
    "p-value",
    "Significance"
]

print(
    tabulate(
        results_summary,
        headers=headers,
        tablefmt="grid",
        floatfmt=".5f"
    )
)


# ============================================================
# 15. Detailed McNemar Tables
# ============================================================

print("\n\n")
print("=" * 90)
print("Detailed 2×2 McNemar Comparison Tables")
print("=" * 90)

for model_name, table in tables.items():

    print(f"\n{model_name}:")

    print(
        tabulate(
            table,
            headers=[
                "Ensemble Correct",
                "Ensemble Wrong"
            ],
            showindex=[
                "Baseline Correct",
                "Baseline Wrong"
            ],
            tablefmt="fancy_grid"
        )
    )


# ============================================================
# 16. Discordant Cases
# ============================================================

print("\n\n")
print("=" * 90)
print("Discordant Cases")
print("=" * 90)

for row in results_summary:

    model_name = row[0]

    b = row[2]
    c = row[3]

    print(f"\n{model_name}")

    print(
        f"Baseline correct / Ensemble wrong = {b}"
    )

    print(
        f"Baseline wrong / Ensemble correct = {c}"
    )

    if c > b:

        print(
            "→ Ensemble corrects more errors than it introduces."
        )

    elif b > c:

        print(
            "→ Ensemble introduces more errors than it corrects."
        )

    else:

        print(
            "→ Both have the same number of discordant cases."
        )

✅ Loaded 24966 predictions from New_CWeighted_Caner.txt


McNemar Test: Baseline Models vs. Proposed Ensemble

Ensemble: New_CWeighted_Caner.txt
Number of ensemble predictions: 24966

------------------------------------------------------------------------------------------
Processing: Camel_msa_predictions_CANER_Corpus.txt
------------------------------------------------------------------------------------------
✅ Loaded 24966 predictions from Camel_msa_predictions_CANER_Corpus.txt
Baseline predictions : 24966
Ensemble predictions : 24966
✅ Token alignment verified
✅ Gold Tag alignment verified

------------------------------------------------------------------------------------------
Processing: Camel_ca_predictions_CANER_Corpus.txt
------------------------------------------------------------------------------------------
✅ Loaded 24966 predictions from Camel_ca_predictions_CANER_Corpus.txt
Baseline predictions : 24966
Ensemble predictions : 24966
✅ Token alignment verified
✅ Gold Ta

#BootStrab

In [ ]:
# ============================================================
# 3. READ PREDICTION FILE
# ============================================================

def read_prediction_file(filepath):
    """
    Reads a prediction file with the format:

    Token    Gold Tag    Predicted Tag

    Blank lines separate sentences.
    """

    sentences = []

    current_sentence = []

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:

        for line in f:

            line = line.rstrip("\n\r")

            # Skip completely empty lines
            if line.strip() == "":
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue

            # Skip header
            if line.strip().lower().startswith("token"):
                continue

            # Split by tabs
            parts = line.split("\t")

            # Remove empty trailing fields
            parts = [p.strip() for p in parts]

            if len(parts) < 3:
                continue

            token = parts[0]
            gold = parts[1]
            pred = parts[2]

            current_sentence.append({
                "token": token,
                "gold": gold,
                "pred": pred
            })

    # Add final sentence
    if current_sentence:
        sentences.append(current_sentence)

    return sentences


In [ ]:
# ============================================================
# 4. READ FILE WITHOUT SENTENCE BOUNDARIES
#    Used for COMPIN files
# ============================================================

def read_ensemble_file(filepath):
    """
    Reads a file with:

    Token    Gold Tag    Predicted Tag

    but without blank-line sentence boundaries.
    """

    rows = []

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:

        for line in f:

            line = line.rstrip("\n\r")

            if line.strip() == "":
                continue

            if line.strip().lower().startswith("token"):
                continue

            parts = line.split("\t")

            parts = [p.strip() for p in parts]

            if len(parts) < 3:
                continue

            rows.append({
                "token": parts[0],
                "gold": parts[1],
                "pred": parts[2]
            })

    return rows


In [ ]:

# ============================================================
# 5. LOAD ALL ANERCorp FILES
# ============================================================

print("Reading prediction files...")

camel_ca_sentences = read_prediction_file(ANER_FILES["Camel-CA"])
camel_mix_sentences = read_prediction_file(ANER_FILES["Camel-Mix"])
camel_msa_sentences = read_prediction_file(ANER_FILES["Camel-MSA"])
Arbert_sentences = read_prediction_file(ANER_FILES["Arbert"])

ensemble1_rows = read_ensemble_file(ANER_FILES["Ensemble-Majority"])
ensemble2_rows = read_ensemble_file(ANER_FILES["Ensemble-Weighted"])


Reading prediction files...


In [ ]:

# ============================================================
# 6. BASIC VALIDATION
# ============================================================

model_sentences = {
    "Camel-MSA": camel_msa_sentences,
    "Camel-CA": camel_ca_sentences,
    "Camel-Mix": camel_mix_sentences,
    "Arbert": Arbert_sentences
}
print("\n========== BASIC FILE CHECK ==========")

for name, sentences in model_sentences.items():

    n_sentences = len(sentences)
    n_tokens = sum(len(s) for s in sentences)

    print(
        f"{name:15s} -> "
        f"{n_sentences} sentences, "
        f"{n_tokens} tokens"
    )

print(
    f"Ensemble-Majority     -> "
    f"{len(ensemble1_rows)} tokens"
)

print(
    f"Ensemble-Weighted      -> "
    f"{len(ensemble2_rows)} tokens"
)




========== BASIC FILE CHECK ==========
Camel-MSA       -> 670 sentences, 24966 tokens
Camel-CA        -> 670 sentences, 24966 tokens
Camel-Mix       -> 670 sentences, 24966 tokens
Arbert          -> 670 sentences, 24966 tokens
Ensemble-Majority     -> 24966 tokens
Ensemble-Weighted      -> 24966 tokens


In [ ]:

# ============================================================
# 7. CHECK THAT ALL MODEL FILES HAVE IDENTICAL
#    SENTENCE/TOKEN/GOLD STRUCTURE
# ============================================================

reference = camel_ca_sentences

for name, sentences in model_sentences.items():

    if len(sentences) != len(reference):
        raise ValueError(
            f"Sentence count mismatch: {name}"
        )

    for i in range(len(reference)):

        ref_tokens = [x["token"] for x in reference[i]]
        cur_tokens = [x["token"] for x in sentences[i]]

        ref_gold = [x["gold"] for x in reference[i]]
        cur_gold = [x["gold"] for x in sentences[i]]

        if ref_tokens != cur_tokens:
            raise ValueError(
                f"Token mismatch in {name}, sentence {i}"
            )

        if ref_gold != cur_gold:
            raise ValueError(
                f"Gold-label mismatch in {name}, sentence {i}"
            )

print("\nAll four baseline prediction files have:")
print("✓ identical sentence boundaries")
print("✓ identical token sequences")
print("✓ identical gold labels")




All four baseline prediction files have:
✓ identical sentence boundaries
✓ identical token sequences
✓ identical gold labels


In [ ]:

# ============================================================
# 8. RECONSTRUCT SENTENCE BOUNDARIES FOR ENSEMBLES
# ============================================================

sentence_lengths = [len(s) for s in reference]

total_tokens = sum(sentence_lengths)

if len(ensemble1_rows) != total_tokens:
    raise ValueError(
        "Ensemble-1 token count does not match baseline files."
    )

if len(ensemble2_rows) != total_tokens:
    raise ValueError(
        "Ensemble-2 token count does not match baseline files."
    )


def reconstruct_sentences(rows, sentence_lengths):

    sentences = []

    start = 0

    for length in sentence_lengths:

        end = start + length

        sentence = rows[start:end]

        sentences.append(sentence)

        start = end

    return sentences


In [ ]:
ensemble1_sentences = reconstruct_sentences(
    ensemble1_rows,
    sentence_lengths
)

ensemble2_sentences = reconstruct_sentences(
    ensemble2_rows,
    sentence_lengths
)


print(
    f"\nEnsemble-Majority reconstructed into "
    f"{len(ensemble1_sentences)} sentences."
)

print(
    f"Ensemble-Weighted reconstructed into "
    f"{len(ensemble2_sentences)} sentences."
)


Ensemble-Majority reconstructed into 670 sentences.
Ensemble-Weighted reconstructed into 670 sentences.


In [ ]:
# ============================================================
# 9. ENTITY EXTRACTION
# ============================================================

def extract_entities(tags):
    """
    Extract entities from BIO/IOB-style NER tags.

    Each entity is represented as:

        (start_token, end_token, entity_type)

    Example:
        B-PER I-PER O
        -> (0, 1, PER)
    """

    entities = []

    start = None
    entity_type = None

    for i, tag in enumerate(tags):

        if tag is None:
            tag = "O"

        tag = tag.strip()

        # ----------------------------------------------------
        # Outside
        # ----------------------------------------------------

        if tag == "O" or tag == "":
            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

                start = None
                entity_type = None

            continue

        # ----------------------------------------------------
        # B-XXX
        # ----------------------------------------------------

        if tag.startswith("B-"):

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = i
            entity_type = tag[2:]

        # ----------------------------------------------------
        # I-XXX
        # ----------------------------------------------------

        elif tag.startswith("I-"):

            current_type = tag[2:]

            # I without active entity
            if start is None:

                start = i
                entity_type = current_type

            # I with different entity type
            elif entity_type != current_type:

                entities.append(
                    (start, i - 1, entity_type)
                )

                start = i
                entity_type = current_type

        # ----------------------------------------------------
        # Other format
        # ----------------------------------------------------

        else:

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = i
            entity_type = tag

    # Close final entity
    if start is not None:

        entities.append(
            (start, len(tags) - 1, entity_type)
        )

    return set(entities)

In [ ]:

# ============================================================
# 10. SENTENCE-LEVEL ENTITY F1
# ============================================================

def calculate_f1_from_sentences(sentences):

    total_gold = 0
    total_pred = 0
    total_correct = 0

    for sentence in sentences:

        gold_tags = [
            x["gold"] for x in sentence
        ]

        pred_tags = [
            x["pred"] for x in sentence
        ]

        gold_entities = extract_entities(gold_tags)
        pred_entities = extract_entities(pred_tags)

        total_gold += len(gold_entities)
        total_pred += len(pred_entities)

        total_correct += len(
            gold_entities.intersection(pred_entities)
        )

    precision = (
        total_correct / total_pred
        if total_pred > 0
        else 0
    )

    recall = (
        total_correct / total_gold
        if total_gold > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return precision, recall, f1



In [ ]:

# ============================================================
# 11. CREATE SENTENCE OBJECTS FOR ALL SYSTEMS
# ============================================================

all_systems = {
    "Camel-MSA": camel_msa_sentences,
    "Camel-CA": camel_ca_sentences,
    "Camel-Mix": camel_mix_sentences,
    "Arbert": Arbert_sentences,
    "Ensemble-Majority": ensemble1_sentences,
    "Ensemble-Weighted": ensemble2_sentences
}



In [ ]:

# ============================================================
# 12. CHECK ORIGINAL F1 RESULTS
# ============================================================

print("\n========== ORIGINAL ENTITY-LEVEL F1 ==========")

original_f1 = {}

for name, sentences in all_systems.items():

    p, r, f1 = calculate_f1_from_sentences(sentences)

    original_f1[name] = f1

    print(
        f"{name:15s} "
        f"P = {p*100:.4f} | "
        f"R = {r*100:.4f} | "
        f"F1 = {f1*100:.4f}"
    )




========== ORIGINAL ENTITY-LEVEL F1 ==========
Camel-MSA       P = 98.0624 | R = 98.7534 | F1 = 98.4067
Camel-CA        P = 98.4851 | R = 98.6631 | F1 = 98.5740
Camel-Mix       P = 98.2718 | R = 98.6269 | F1 = 98.4491
Arbert          P = 98.0645 | R = 98.8618 | F1 = 98.4615
Ensemble-Majority P = 98.3285 | R = 98.8437 | F1 = 98.5855
Ensemble-Weighted P = 98.5053 | R = 98.8257 | F1 = 98.6652


In [ ]:

# ============================================================
# 13. PAIRED SENTENCE-LEVEL BOOTSTRAP
# ============================================================

def paired_bootstrap_f1(
    ensemble_sentences,
    baseline_sentences,
    n_bootstrap=1000,
    seed=1
):
    """
    Paired sentence-level bootstrap.

    Resampling unit:
        Sentence

    Metric:
        Entity-level F1

    Difference:
        F1(ensemble) - F1(baseline)

    Same sampled sentence indices are used for
    ensemble and baseline.

    95% CI:
        Percentile bootstrap CI
        [2.5%, 97.5%]
    """

    rng = np.random.default_rng(seed)

    n_sentences = len(ensemble_sentences)

    observed_p_e, observed_r_e, observed_f1_e = \
        calculate_f1_from_sentences(ensemble_sentences)

    observed_p_b, observed_r_b, observed_f1_b = \
        calculate_f1_from_sentences(baseline_sentences)

    observed_difference = (
        observed_f1_e - observed_f1_b
    )

    bootstrap_differences = []

    for iteration in range(n_bootstrap):

        # ----------------------------------------------------
        # Sample sentences WITH replacement
        # ----------------------------------------------------

        sampled_indices = rng.integers(
            0,
            n_sentences,
            size=n_sentences
        )

        sampled_ensemble = [
            ensemble_sentences[i]
            for i in sampled_indices
        ]

        sampled_baseline = [
            baseline_sentences[i]
            for i in sampled_indices
        ]

        # ----------------------------------------------------
        # Calculate F1 on the same sampled sentences
        # ----------------------------------------------------

        _, _, f1_e = calculate_f1_from_sentences(
            sampled_ensemble
        )

        _, _, f1_b = calculate_f1_from_sentences(
            sampled_baseline
        )

        # ----------------------------------------------------
        # Paired F1 difference
        # ----------------------------------------------------

        difference = f1_e - f1_b

        bootstrap_differences.append(
            difference
        )

    bootstrap_differences = np.array(
        bootstrap_differences
    )

    # --------------------------------------------------------
    # Mean bootstrap difference
    # --------------------------------------------------------

    mean_difference = np.mean(
        bootstrap_differences
    )

    # --------------------------------------------------------
    # 95% percentile confidence interval
    # --------------------------------------------------------

    ci_low, ci_high = np.percentile(
        bootstrap_differences,
        [2.5, 97.5]
    )

    # --------------------------------------------------------
    # Significance
    # --------------------------------------------------------

    significant = (
        ci_low > 0 or ci_high < 0
    )

    significance = (
        "Significant (p < 0.05)"
        if significant
        else "Not significant"
    )

    return {
        "Observed Delta F1": observed_difference,
        "Mean Delta F1": mean_difference,
        "CI Low": ci_low,
        "CI High": ci_high,
        "Significance": significance
    }



In [ ]:

# ============================================================
# 14. RUN BOOTSTRAP FOR ONE ENSEMBLE
# ============================================================

def run_ensemble_bootstrap(
    ensemble_name,
    baseline_names,
    all_systems,
    n_bootstrap=1000,
    seed=1
):

    results = []

    for baseline_name in baseline_names:

        result = paired_bootstrap_f1(
            ensemble_sentences=all_systems[ensemble_name],
            baseline_sentences=all_systems[baseline_name],
            n_bootstrap=n_bootstrap,
            seed=seed
        )

        results.append({
            "Baseline Model": baseline_name,
            "Observed ΔF1": result["Observed Delta F1"],
            "Mean ΔF1": result["Mean Delta F1"],
            "95 % CI Low": result["CI Low"],
            "95 % CI High": result["CI High"],
            "Significance": result["Significance"]
        })

    return pd.DataFrame(results)


# ============================================================
# 15. ANERCorp BOOTSTRAP TABLES
# ============================================================

aner_baselines = [
    "Camel-MSA",
    "Camel-CA",
    "Camel-Mix",
    "Arbert"
]


print("\n==========================================")
print("CANER Corpus - ENSEMBLE Majority")
print("==========================================")

aner_ensemble1 = run_ensemble_bootstrap(
    "Ensemble-Majority",
    aner_baselines,
    all_systems,
    n_bootstrap=1000,
    seed=1
)

print(aner_ensemble1.to_string(index=False))


print("\n==========================================")
print("CANER corpus - ENSEMBLE Weighted")
print("==========================================")

aner_ensemble2 = run_ensemble_bootstrap(
    "Ensemble-Weighted",
    aner_baselines,
    all_systems,
    n_bootstrap=1000,
    seed=1
)

print(aner_ensemble2.to_string(index=False))


CANER Corpus - ENSEMBLE Majority
Baseline Model  Observed ΔF1  Mean ΔF1  95 % CI Low  95 % CI High           Significance
     Camel-MSA      0.001788  0.001764     0.000255      0.003430 Significant (p < 0.05)
      Camel-CA      0.000115  0.000098    -0.001730      0.002097        Not significant
     Camel-Mix      0.001364  0.001403    -0.000426      0.003283        Not significant
        Arbert      0.001239  0.001254    -0.000476      0.003177        Not significant

CANER corpus - ENSEMBLE Weighted
Baseline Model  Observed ΔF1  Mean ΔF1  95 % CI Low  95 % CI High           Significance
     Camel-MSA      0.002585  0.002578     0.000698      0.004657 Significant (p < 0.05)
      Camel-CA      0.000912  0.000912    -0.000453      0.002483        Not significant
     Camel-Mix      0.002162  0.002216     0.000453      0.004093 Significant (p < 0.05)
        Arbert      0.002037  0.002067     0.000237      0.003930 Significant (p < 0.05)


In [ ]:

# ============================================================
# 16. FORMAT TABLES EXACTLY LIKE PAPER
# ============================================================

def format_bootstrap_table(df):

    formatted = df.copy()

    formatted["Mean ΔF1"] = formatted[
        "Mean ΔF1"
    ].map(lambda x: f"{x:.5f}")

    formatted["95 % CI Low"] = formatted[
        "95 % CI Low"
    ].map(lambda x: f"{x:.5f}")

    formatted["95 % CI High"] = formatted[
        "95 % CI High"
    ].map(lambda x: f"{x:.5f}")

    formatted["Observed ΔF1"] = formatted[
        "Observed ΔF1"
    ].map(lambda x: f"{x:.5f}")

    return formatted


print("\n\n====================================================")
print("TABLE 10 - BOOTSTRAP ANALYSIS")
print("====================================================")

print("\n(c) Bootstrap analysis for Ensemble-1")
print(
    format_bootstrap_table(
        aner_ensemble1
    ).to_string(index=False)
)

print("\n(d) Bootstrap analysis for Ensemble-2")
print(
    format_bootstrap_table(
        aner_ensemble2
    ).to_string(index=False)
)




TABLE 10 - BOOTSTRAP ANALYSIS

(c) Bootstrap analysis for Ensemble-1
Baseline Model Observed ΔF1 Mean ΔF1 95 % CI Low 95 % CI High           Significance
     Camel-MSA      0.00179  0.00176     0.00025      0.00343 Significant (p < 0.05)
      Camel-CA      0.00011  0.00010    -0.00173      0.00210        Not significant
     Camel-Mix      0.00136  0.00140    -0.00043      0.00328        Not significant
        Arbert      0.00124  0.00125    -0.00048      0.00318        Not significant

(d) Bootstrap analysis for Ensemble-2
Baseline Model Observed ΔF1 Mean ΔF1 95 % CI Low 95 % CI High           Significance
     Camel-MSA      0.00259  0.00258     0.00070      0.00466 Significant (p < 0.05)
      Camel-CA      0.00091  0.00091    -0.00045      0.00248        Not significant
     Camel-Mix      0.00216  0.00222     0.00045      0.00409 Significant (p < 0.05)
        Arbert      0.00204  0.00207     0.00024      0.00393 Significant (p < 0.05)


In [ ]:


# ============================================================
# 17. SAVE RESULTS TO EXCEL
# ============================================================

output_excel = "/mnt/data/ANERCorp_Bootstrap_Results.xlsx"

with pd.ExcelWriter(
    output_excel,
    engine="openpyxl"
) as writer:

    aner_ensemble1.to_excel(
        writer,
        sheet_name="Ensemble-1",
        index=False
    )

    aner_ensemble2.to_excel(
        writer,
        sheet_name="Ensemble-2",
        index=False
    )

print(
    f"\nResults saved to:\n{output_excel}"
)


# ============================================================
# 18. SAVE RESULTS TO CSV
# ============================================================

aner_ensemble1.to_csv(
    "/mnt/data/ANERCorp_Ensemble1_Bootstrap.csv",
    index=False
)

aner_ensemble2.to_csv(
    "/mnt/data/ANERCorp_Ensemble2_Bootstrap.csv",
    index=False
)

print("CSV files saved successfully.")



In [ ]:

# ============================================================
# 19. FINAL METHODOLOGY SUMMARY
# ============================================================

print("\n====================================================")
print("BOOTSTRAP METHODOLOGY")
print("====================================================")

print("""
Resampling unit:
    Test-set sentences

Number of bootstrap iterations:
    1000

Bootstrap type:
    Paired bootstrap

Metric:
    Entity-level F1-score

Difference tested:
    F1(ensemble) - F1(baseline)

Confidence interval:
    95% percentile bootstrap CI

CI limits:
    2.5th and 97.5th percentiles

Significance criterion:
    Significant if the 95% CI does not include zero.
""")